In [1]:
%pip install pyspark==3.5.1
%pip install pandas

  Using cached pyspark-3.5.1-py2.py3-none-any.whl
  Using cached py4j-0.10.9.7-py2.py3-none-any.whl.metadata (1.5 kB)
Using cached py4j-0.10.9.7-py2.py3-none-any.whl (200 kB)
Note: you may need to restart the kernel to use updated packages.
Note: you may need to restart the kernel to use updated packages.


In [2]:
%pip install findspark

  Using cached findspark-2.0.1-py2.py3-none-any.whl.metadata (352 bytes)
Using cached findspark-2.0.1-py2.py3-none-any.whl (4.4 kB)
Note: you may need to restart the kernel to use updated packages.


In [5]:
from pyspark.sql import SparkSession
import findspark
from pyspark.sql import SparkSession

try:
    spark.stop()
except:
    pass

scala_version = '2.12'
spark_version = '3.5.1'
packages = [ f'org.apache.spark:spark-sql-kafka-0-10_{scala_version}:{spark_version}' , 'org.apache.kafka:kafka-clients:3.5.1' ]

findspark.init()
spark = SparkSession.builder.master("local").appName("kafka").config("spark.jars.packages", ",".join(packages)).getOrCreate()


spark.sparkContext.setLogLevel("ERROR")


In [6]:
topic_name = 'FloodPrediction'
kafka_server = 'localhost:9092'

df_raw = spark.read.format("kafka").option("kafka.bootstrap.servers", kafka_server).option("subscribe", topic_name).load()

In [7]:
df_raw .printSchema()
df_raw.toPandas()

root
 |-- key: binary (nullable = true)
 |-- value: binary (nullable = true)
 |-- topic: string (nullable = true)
 |-- partition: integer (nullable = true)
 |-- offset: long (nullable = true)
 |-- timestamp: timestamp (nullable = true)
 |-- timestampType: integer (nullable = true)



,key,value,topic,partition,offset,timestamp,timestampType
0,None,"[123, 34, 115, 121, 115, 116, 101, 109, 58, 10...",FloodPrediction,0,0,2025-09-30 19:21:46.638,0
1,None,"[123, 34, 115, 121, 115, 116, 101, 109, 58, 10...",FloodPrediction,0,1,2025-09-30 19:21:51.644,0
2,None,"[123, 34, 115, 121, 115, 116, 101, 109, 58, 10...",FloodPrediction,0,2,2025-09-30 19:21:56.651,0
3,None,"[123, 34, 115, 121, 115, 116, 101, 109, 58, 10...",FloodPrediction,0,3,2025-09-30 19:23:33.407,0
4,None,"[123, 34, 115, 121, 115, 116, 101, 109, 58, 10...",FloodPrediction,0,4,2025-09-30 19:23:38.413,0
5,None,"[123, 34, 115, 121, 115, 116, 101, 109, 58, 10...",FloodPrediction,0,5,2025-09-30 19:23:43.419,0
6,None,"[123, 34, 115, 121, 115, 116, 101, 109, 58, 10...",FloodPrediction,0,6,2025-09-30 19:23:48.426,0
7,None,"[123, 34, 115, 121, 115, 116, 101, 109, 58, 10...",FloodPrediction,0,7,2025-09-30 19:23:53.432,0


In [8]:
from time import sleep
from IPython.display import display, clear_output
from pyspark.sql.functions import from_json, col
from pyspark.sql.types import *

input_schema = StructType([
    StructField("system_index", IntegerType(), True),
    StructField("date", StringType(), True),  # or DateType() if you parse it later
    StructField("rainfall_3day_cumulative_mm", DoubleType(), True),
    StructField("rainfall_5day_cumulative_mm", DoubleType(), True),
    StructField("rainfall_7day_cumulative_mm", DoubleType(), True),
    StructField("rainfall_max_mm", DoubleType(), True),
    StructField("rainfall_mean_mm", DoubleType(), True),
    StructField("rainfall_std_mm", DoubleType(), True),
    StructField("soil_moisture_top10cm_mm", DoubleType(), True),
    StructField("subsurface_runoff_mm", DoubleType(), True),
    StructField("surface_runoff_mm", DoubleType(), True),
    StructField("geo", StringType(), True)
])

df_input = df_raw.selectExpr("CAST(value AS STRING) as json") \
    .select(from_json(col("json"), input_schema).alias("data")) \
    .select("data.*")


for i in range(0, 100):
    try:
        print("Showing live view refreshed every 5 seconds")
        print(f"Seconds passed: {i*5}")
        display(df_input.toPandas())
        sleep(5)
        clear_output(wait=True)
    except KeyboardInterrupt:
        print("break")
        break

print("Live view ended...")

Showing live view refreshed every 5 seconds
Seconds passed: 15


,system_index,date,rainfall_3day_cumulative_mm,rainfall_5day_cumulative_mm,rainfall_7day_cumulative_mm,rainfall_max_mm,rainfall_mean_mm,rainfall_std_mm,soil_moisture_top10cm_mm,subsurface_runoff_mm,surface_runoff_mm,geo
0,NaN,11/1/1984,152.642107,152.642107,152.642107,244.755600,152.959280,33.956949,-9999.0,-9999.0,-9999.0,None
1,NaN,11/2/1984,210.656282,210.656282,210.656282,125.197693,58.868336,28.436351,-9999.0,-9999.0,-9999.0,None
2,NaN,11/3/1984,220.012695,220.012695,220.012695,37.429073,9.759217,11.832672,-9999.0,-9999.0,-9999.0,None
3,NaN,11/1/1984,152.642107,152.642107,152.642107,244.755600,152.959280,33.956949,-9999.0,-9999.0,-9999.0,None
4,NaN,11/2/1984,210.656282,210.656282,210.656282,125.197693,58.868336,28.436351,-9999.0,-9999.0,-9999.0,None
5,NaN,11/3/1984,220.012695,220.012695,220.012695,37.429073,9.759217,11.832672,-9999.0,-9999.0,-9999.0,None
6,NaN,11/4/1984,67.457487,220.099593,220.099593,16.363964,0.149993,1.835437,-9999.0,-9999.0,-9999.0,None
7,NaN,11/5/1984,9.443311,220.099593,220.099593,0.000000,0.000000,0.000000,-9999.0,-9999.0,-9999.0,None
8,NaN,11/6/1984,0.086899,67.457487,220.099593,0.000000,0.000000,0.000000,-9999.0,-9999.0,-9999.0,None
9,NaN,11/7/1984,129.635518,139.078829,349.735111,221.834518,131.957792,30.882843,-9999.0,-9999.0,-9999.0,None


break
Live view ended...


In [ ]:
%pip install tensorflow==2.12
%pip install joblib
%pip install scikit-learn

  Using cached idna-3.10-py3-none-any.whl.metadata (10 kB)
  Using cached certifi-2025.4.26-py3-none-any.whl.metadata (2.5 kB)
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 479.6/479.6 MB 21.5 MB/s eta 0:00:0000:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 6.0/6.0 MB 17.2 MB/s eta 0:00:00a 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 5.3/5.3 MB 15.9 MB/s eta 0:00:00a 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.7/1.7 MB 16.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 24.5/24.5 MB 18.4 MB/s eta 0:00:00a 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 17.3/17.3 MB 18.2 MB/s eta 0:00:00a 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 5.6/5.6 MB 17.5 MB/s eta 0:00:00a 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.4/2.4 MB 17.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 6.6/6.6 MB 18.3 MB/s eta 0:00:00a 0:00:01
Using cached certifi-2025.4.26-py3-none-any.whl (159 kB)
Using cached idna-3.10-py3-none-any.whl (70 kB)
  A

In [18]:
%pip install pyarrow
%pip install joblib
%pip install torch torchvision torchaudio --index-url https://download.pytorch.org/whl/cu121
%pip install pandas
%pip install -U scikit-learn

Note: you may need to restart the kernel to use updated packages.
Note: you may need to restart the kernel to use updated packages.
Looking in indexes: https://download.pytorch.org/whl/cu121
Note: you may need to restart the kernel to use updated packages.
Note: you may need to restart the kernel to use updated packages.
Note: you may need to restart the kernel to use updated packages.


In [19]:
import torch.nn as nn

DROPOUT_RATE = 0.4  # dropout rate
# -----------------------------------------
# Define the CNN model
# -----------------------------------------
class CNN1DClassifier(nn.Module):
    def __init__(self, in_channels, sequence_length):
        super().__init__()
        self.conv_block = nn.Sequential(
            nn.Conv1d(in_channels=in_channels, out_channels=32, kernel_size=3, padding=1),
            nn.BatchNorm1d(32),
            nn.ReLU(),
            nn.Dropout(DROPOUT_RATE),

            nn.Conv1d(32, 64, kernel_size=3, padding=1),
            nn.BatchNorm1d(64),
            nn.ReLU(),
            nn.Dropout(DROPOUT_RATE),
        )
        self.flatten = nn.Flatten()
        self.fc = nn.Sequential(
            nn.Linear(64 * sequence_length, 64),
            nn.ReLU(),
            nn.Linear(64, 1)  # No sigmoid
        )

    def forward(self, x):
        x = self.conv_block(x)
        x = self.flatten(x)
        return self.fc(x)


In [29]:
import numpy as np
import pandas as pd
import joblib
import torch

from pyspark.sql.functions import pandas_udf
from pyspark.sql.types import DoubleType

# Define once outside if known
WINDOW_SIZE = 2  # use past 2 days
FEATURES = ["rainfall_3day_cumulative_mm",
    "rainfall_5day_cumulative_mm",
    "rainfall_7day_cumulative_mm",
    "rainfall_max_mm",
    "rainfall_mean_mm",
    "rainfall_std_mm",
    "soil_moisture_top10cm_mm",
    "subsurface_runoff_mm",
    "surface_runoff_mm"]
    
base_features = FEATURES
num_features_per_step = len(FEATURES)

# -------------------------------
# 3. Load model + scaler (broadcasted once per worker)
# -------------------------------
scaler = joblib.load("scaler.pkl")

model = CNN1DClassifier(num_features_per_step, WINDOW_SIZE)
model.load_state_dict(torch.load("weighted_data_cnn_model.pt", map_location="cpu"))
model.eval()

# -------------------------------
# 4. Helper: reshape to CNN input
# -------------------------------
def reshape_to_cnn_input(X):
    return X.reshape((-1, num_features_per_step, WINDOW_SIZE))

@pandas_udf(DoubleType())
def predict_flood_udf(*cols: pd.Series) -> pd.Series:
    # Reconstruct DataFrame from feature columns
    pdf = pd.concat(cols, axis=1)
    pdf.columns = FEATURES   # raw base features, not lagged

    # Build lag features with pandas shift
    for lag in range(1, WINDOW_SIZE + 1):
        for col in FEATURES:
            pdf[f"{col}_lag{lag}"] = pdf[col].shift(lag)

    # Drop rows with NaN lag values
    pdf = pdf.dropna().reset_index(drop=True)
    if pdf.empty:
        return pd.Series([0.0] * len(cols[0]))

    # Keep only lag features (as in training)
    lag_features = [f"{col}_lag{lag}" for col in FEATURES for lag in range(1, WINDOW_SIZE + 1)]
    X = pdf[lag_features].values   # shape (n_samples, num_features*WINDOW_SIZE)

    X_scaled = scaler.transform(X)  # still 2D
    X_seq = reshape_to_cnn_input(X_scaled)  # (batch, features, window)

    X_tensor = torch.tensor(np.array(X_seq), dtype=torch.float32)
    with torch.no_grad():
        preds = model(X_tensor).squeeze().numpy()

    # Align prediction length back to input rows
    # First WINDOW_SIZE rows cannot be predicted
    preds_aligned = [np.nan] * WINDOW_SIZE + preds.tolist()
    return pd.Series(preds_aligned)
  # shift since window=2

/tmp/ipykernel_13708/2790312030.py:30: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  model.load_state_dict(torch.load("weighted_data_cnn_model.pt", map_location="cpu"))


In [30]:
from IPython.display import display, clear_output
from pyspark.sql.functions import from_json, col
from pyspark.sql.types import *

df_input = df_raw.selectExpr("CAST(value AS STRING) as json") \
    .select(from_json(col("json"), input_schema).alias("data")) \
    .select("data.*")

df_sample = df_input.limit(1000)
df_pred = df_sample.withColumn(
    "flood_pred",
    predict_flood_udf(*[col(c) for c in base_features])
)

df_input_clean = df_pred.limit(50).select("flood_pred")
df_input_clean = df_input_clean.na.drop()
df_input_clean.show()

/home/kaos/AfterGradEx/time_series/flood_predicition/.conda/lib/python3.8/site-packages/sklearn/base.py:465: UserWarning: X does not have valid feature names, but StandardScaler was fitted with feature names
  warnings.warn(


+-----------------+
|       flood_pred|
+-----------------+
|-10215.8681640625|
|-10314.5302734375|
|-10061.6884765625|
|-10215.8681640625|
|-10314.5302734375|
|-10334.5361328125|
|-10347.6416015625|
|-10348.3056640625|
| -10079.951171875|
|-10054.5009765625|
| -10215.869140625|
+-----------------+

